# SHAP Interpretation for the ICU Mortality Random Forest — src Refactor

This notebook explains the underlying **uncalibrated Random Forest** using the reusable SHAP and feature-importance utilities in `src.interpretation`.

It produces global feature importance, a SHAP beeswarm, dependence plots, local patient explanations, and a comparison with Random Forest built-in importance.

## 1. Bootstrap project imports

In [ ]:
from pathlib import Path
import sys

# Make imports work whether VS Code starts the notebook from project root
# or from the notebooks/ directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "src").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing src/. "
        "Open this notebook from the clinical-outcome-prediction project."
    )

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project root:", PROJECT_DIR)
print("Python:", sys.executable)

## 2. Imports

In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from src.config import (
    MODELS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    EXPLANATIONS_DIR,
    SELECTED_MODEL_CONFIG_PATH,
)
from src.data import (
    load_modeling_splits,
    load_feature_config,
    validate_patient_split,
)
from src.evaluation import (
    classification_metrics,
)
from src.interpretation import (
    compute_tree_shap,
    create_local_explanation,
    global_shap_importance,
    random_forest_builtin_importance,
    compare_feature_importance,
)

THRESHOLD = 0.50
TOP_FEATURES = 20
TOP_LOCAL_FEATURES = 15

RF_MODEL_PATH = MODELS_DIR / "random_forest.joblib"

for directory in [
    TABLES_DIR,
    FIGURES_DIR,
    PREDICTIONS_DIR,
    EXPLANATIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

## 3. Load data, feature configuration, and Random Forest

In [ ]:
train_df, validation_df, test_df = load_modeling_splits()
validate_patient_split(
    train_df,
    validation_df,
    test_df,
)

feature_config = load_feature_config()
random_forest_model = joblib.load(
    RF_MODEL_PATH
)

target_column = feature_config["target_column"]
feature_columns = feature_config["retained_feature_columns"]

selected_model_name = None
if SELECTED_MODEL_CONFIG_PATH.exists():
    with open(
        SELECTED_MODEL_CONFIG_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        selected_model_name = json.load(file).get(
            "selected_model_name"
        )

X_test = test_df[feature_columns].copy()
y_test = test_df[target_column].copy()

test_probability = (
    random_forest_model.predict_proba(
        X_test
    )[:, 1]
)

print("Notebook 7 selected:", selected_model_name)
print("Explained model: uncalibrated Random Forest")
print(
    pd.Series(
        classification_metrics(
            y_test,
            test_probability,
            threshold=THRESHOLD,
        )
    )
)

## 4. Recover fitted preprocessor, classifier, and transformed features

In [ ]:
if not hasattr(
    random_forest_model,
    "named_steps",
):
    raise TypeError(
        "Saved Random Forest is expected to be a scikit-learn Pipeline."
    )

preprocessor = random_forest_model.named_steps[
    "preprocessor"
]
rf_classifier = random_forest_model.named_steps[
    "classifier"
]

X_test_transformed = preprocessor.transform(
    X_test
)
feature_names = (
    preprocessor.get_feature_names_out()
)

X_test_transformed_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index,
)

print("Transformed test shape:", X_test_transformed_df.shape)

## 5. Compute positive-class Tree SHAP values

In [ ]:
explainer, shap_values, base_value = (
    compute_tree_shap(
        rf_classifier,
        X_test_transformed,
    )
)

print("SHAP shape:", shap_values.shape)
print("Positive-class base value:", base_value)

reconstructed = (
    base_value
    + shap_values.sum(axis=1)
)

maximum_additivity_error = float(
    np.max(
        np.abs(
            reconstructed
            - test_probability
        )
    )
)

print(
    "Maximum SHAP additivity error:",
    maximum_additivity_error,
)

## 6. Global SHAP importance

In [ ]:
shap_importance = global_shap_importance(
    shap_values,
    feature_names,
)

shap_importance.head(
    TOP_FEATURES
)

## 7. Global SHAP importance bar chart

In [ ]:
top_importance = (
    shap_importance
    .head(TOP_FEATURES)
    .sort_values(
        "mean_absolute_shap"
    )
)

ax = top_importance.plot(
    x="feature",
    y="mean_absolute_shap",
    kind="barh",
    legend=False,
    figsize=(10, 8),
)

ax.set_title(
    "Random Forest: Global SHAP Importance"
)
ax.set_xlabel(
    "Mean absolute SHAP value"
)
ax.set_ylabel(
    "Transformed feature"
)

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "shap_global_importance_bar.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. SHAP beeswarm

In [ ]:
plt.figure(figsize=(10, 8))

shap.summary_plot(
    shap_values,
    X_test_transformed_df,
    max_display=TOP_FEATURES,
    show=False,
)

plt.title(
    "Random Forest SHAP Summary: Test Cohort"
)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "shap_summary_beeswarm.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 9. Dependence plots for top numeric features

In [ ]:
dependence_features = [
    feature
    for feature in shap_importance["feature"]
    if feature.startswith("numeric__")
    and "missingindicator" not in feature.lower()
][:4]

for feature in dependence_features:
    feature_index = list(
        feature_names
    ).index(feature)

    plt.figure(figsize=(8, 6))

    shap.dependence_plot(
        feature_index,
        shap_values,
        X_test_transformed_df,
        interaction_index=None,
        show=False,
    )

    readable_name = (
        feature.replace(
            "numeric__",
            "",
        )
    )

    plt.title(
        f"SHAP Dependence: {readable_name}"
    )
    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR
        / f"shap_dependence_{readable_name}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

print("Dependence features:", dependence_features)

## 10. Create prediction categories

In [ ]:
test_prediction = (
    test_probability >= THRESHOLD
).astype(int)

prediction_frame = test_df[
    ["subject_id", "hadm_id", "stay_id", target_column]
].copy()

prediction_frame[
    "predicted_probability"
] = test_probability

prediction_frame[
    "predicted_label"
] = test_prediction

prediction_frame[
    "prediction_category"
] = np.select(
    [
        (prediction_frame[target_column] == 1)
        & (prediction_frame["predicted_label"] == 1),

        (prediction_frame[target_column] == 0)
        & (prediction_frame["predicted_label"] == 1),

        (prediction_frame[target_column] == 1)
        & (prediction_frame["predicted_label"] == 0),

        (prediction_frame[target_column] == 0)
        & (prediction_frame["predicted_label"] == 0),
    ],
    [
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative",
    ],
    default="unknown",
)

prediction_frame[
    "prediction_category"
].value_counts()

## 11. Representative patient waterfall plots

In [ ]:
def choose_representative_index(category):
    subset = prediction_frame.loc[
        prediction_frame[
            "prediction_category"
        ] == category
    ]

    if subset.empty:
        return None

    ascending = category in {
        "false_negative",
        "true_negative",
    }

    return int(
        subset.sort_values(
            "predicted_probability",
            ascending=ascending,
        ).index[0]
    )


local_tables = []

for category in [
    "true_positive",
    "false_positive",
    "false_negative",
    "true_negative",
]:
    row_index = choose_representative_index(
        category
    )

    if row_index is None:
        print(
            f"No {category.replace('_', ' ')} available."
        )
        continue

    row_position = X_test.index.get_loc(
        row_index
    )

    explanation = create_local_explanation(
        shap_values,
        X_test_transformed,
        feature_names,
        row_position=row_position,
        base_value=base_value,
    )

    metadata = prediction_frame.loc[
        row_index
    ]

    plt.figure(figsize=(10, 8))
    shap.plots.waterfall(
        explanation,
        max_display=TOP_LOCAL_FEATURES,
        show=False,
    )

    plt.title(
        f"SHAP Waterfall: {category.replace('_', ' ').title()}\\n"
        f"Stay {metadata['stay_id']} | "
        f"Observed {int(metadata[target_column])} | "
        f"Predicted risk {metadata['predicted_probability']:.3f}"
    )

    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR
        / f"shap_waterfall_{category}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

    local_table = pd.DataFrame(
        {
            "feature": feature_names,
            "transformed_feature_value": X_test_transformed[
                row_position
            ],
            "shap_value": shap_values[
                row_position
            ],
        }
    )

    local_table[
        "absolute_shap_value"
    ] = np.abs(
        local_table["shap_value"]
    )

    local_table[
        "prediction_category"
    ] = category

    local_table[
        "stay_id"
    ] = metadata["stay_id"]

    local_tables.append(
        local_table.sort_values(
            "absolute_shap_value",
            ascending=False,
        )
    )

## 12. Compare SHAP importance with Random Forest built-in importance

In [ ]:
rf_importance = (
    random_forest_builtin_importance(
        rf_classifier,
        feature_names,
    )
)

importance_comparison = (
    compare_feature_importance(
        shap_importance,
        rf_importance,
    )
)

importance_comparison.head(
    TOP_FEATURES
)

## 13. Save interpretation outputs

In [ ]:
shap_importance.to_csv(
    TABLES_DIR / "shap_global_importance.csv",
    index=False,
)

importance_comparison.to_csv(
    TABLES_DIR / "shap_vs_random_forest_importance.csv",
    index=False,
)

prediction_frame.to_csv(
    PREDICTIONS_DIR / "test_prediction_categories_for_shap.csv",
    index=False,
)

if local_tables:
    patient_contributions = pd.concat(
        local_tables,
        ignore_index=True,
    )

    patient_contributions.to_csv(
        TABLES_DIR / "shap_patient_contributions.csv",
        index=False,
    )

np.save(
    EXPLANATIONS_DIR / "random_forest_test_shap_values.npy",
    shap_values,
)

X_test_transformed_df.to_csv(
    EXPLANATIONS_DIR / "random_forest_test_transformed_features.csv",
    index=False,
)

with open(
    EXPLANATIONS_DIR / "shap_interpretation_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "explained_model": "Random Forest",
            "explained_class": f"{target_column} = 1",
            "classification_threshold": THRESHOLD,
            "test_rows_explained": int(len(X_test)),
            "feature_count": int(len(feature_names)),
            "base_value": float(base_value),
            "maximum_additivity_error": maximum_additivity_error,
            "warning": (
                "SHAP describes predictive contribution, not causality."
            ),
        },
        file,
        indent=2,
    )

print("Notebook 9 outputs saved.")

## Notebook 9 summary

The notebook now delegates reusable SHAP and feature-importance calculations to `src.interpretation`, while keeping plots and clinical interpretation visible in the notebook.

Next: **Notebook 10 — Threshold, subgroup, clinical utility, and error analysis**.